## Import library

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchinfo import summary

import pandas as pd
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import copy

## Build Resnet 6

In [2]:
# Define the basic block for ResNet
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        # out = torch.add(out, self.shortcut(x))
        out += self.shortcut(x)
        return torch.relu(out)

# Define ResNet-6
class ResNet6(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet6, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.layer1 = self._make_layer(16, 16, 1, stride=2)
        self.layer2 = self._make_layer(16, 32, 1, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(32, num_classes)

    def _make_layer(self, in_channels, out_channels, blocks, stride):
        layers = []
        layers.append(BasicBlock(in_channels, out_channels, stride))
        for _ in range(1, blocks):
            layers.append(BasicBlock(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        out = self.fc(out)
        return out

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNet6().to(device)

In [5]:
model_path = 'resnet6.pth'
model.load_state_dict(torch.load(model_path, weights_only=True))

<All keys matched successfully>

## Pytorch Build in Quantized Model

In [6]:
# Define the basic block for ResNet
class QuantizedBasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(QuantizedBasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.add = torch.nn.quantized.FloatFunctional()
        # self.quant = torch.quantization.QuantStub()
        # self.dequant_1 = torch.quantization.DeQuantStub()
        # self.dequant_2 = torch.quantization.DeQuantStub()

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.add.add(out, self.shortcut(x))
        return torch.relu(out)

# Define ResNet-6
class QuantizedResNet6(nn.Module):
    def __init__(self, num_classes=10):
        super(QuantizedResNet6, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.layer1 = self._make_layer(16, 16, 1, stride=2)
        self.layer2 = self._make_layer(16, 32, 1, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(32, num_classes)
        self.quant = torch.quantization.QuantStub()
        self.dequant = torch.quantization.DeQuantStub()

    def _make_layer(self, in_channels, out_channels, blocks, stride):
        layers = []
        layers.append(QuantizedBasicBlock(in_channels, out_channels, stride))
        for _ in range(1, blocks):
            layers.append(QuantizedBasicBlock(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.quant(x)
        out = torch.relu(self.bn1(self.conv1(out)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        out = self.fc(out)
        out = self.dequant(out)
        return out

In [7]:
# Wrap and prepare the model for quantization
quantized_model = QuantizedResNet6()

# Load Model
model_path = 'resnet6.pth'
quantized_model.load_state_dict(torch.load(model_path, weights_only=True))


<All keys matched successfully>

## Set up the dataset and dataloader

In [8]:
# Set up the dataset and dataloaders
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False, num_workers=2)

Files already downloaded and verified
Files already downloaded and verified


## Define test function

In [9]:
def test(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f"Test Accuracy: {100 * correct / total:.2f}%")

In [10]:
def test_first_images(model, test_loader, num_images=192):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        count = 0
        for images, labels in test_loader:
            if count >= num_images:
                break
            remain = num_images - count
            if remain > 100:
                images, labels = images.to(device), labels.to(device)
            else:
                images, labels = images[:remain].to(device), labels[:remain].to(device)
            
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            count = count + 100
            
        print(f"Total: {total}, Test Accuracy: {100 * correct / total:.2f}%")

## Quantize Model

In [11]:
quantized_model.train()
backend = "fbgemm"
torch.backends.quantized.engine = backend
quantized_model.qconfig = torch.quantization.get_default_qconfig(backend)

# Prepare the model for static quantization
torch.quantization.prepare_qat(quantized_model, inplace=True)

# Calibrate the model using the test dataset
with torch.no_grad():
    for images, _ in test_loader:  # Ensure test_loader is properly defined
        quantized_model(images)

# Convert the model to INT8
torch.quantization.convert(quantized_model, inplace=True)


c:\Users\user\anaconda3\Lib\site-packages\torch\ao\quantization\observer.py:220: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


QuantizedResNet6(
  (conv1): QuantizedConv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), scale=0.029561670497059822, zero_point=62, padding=(1, 1), bias=False)
  (bn1): QuantizedBatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer1): Sequential(
    (0): QuantizedBasicBlock(
      (conv1): QuantizedConv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), scale=0.18188239634037018, zero_point=89, padding=(1, 1), bias=False)
      (bn1): QuantizedBatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): QuantizedConv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), scale=0.08419828861951828, zero_point=65, padding=(1, 1), bias=False)
      (bn2): QuantizedBatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (add): QFunctional(
        scale=0.13724027574062347, zero_point=71
        (activation_post_process): Identity()
      )
      (shortcut): Sequential(
        (0): QuantizedConv2d(16, 16, ke

In [12]:
test_first_images(model, test_loader, num_images=64)
test_first_images(quantized_model, test_loader, num_images=64)

Total: 64, Test Accuracy: 67.19%
Total: 64, Test Accuracy: 70.31%


## Get the parameters from layers

In [13]:
# Get the quantization parameters of the QuantStub
print("QuantStub scale:", quantized_model.quant.scale)
print("QuantStub zero_point:", quantized_model.quant.zero_point)
print("QuantStub dtype:", quantized_model.quant.dtype)

QuantStub scale: tensor([0.0157])
QuantStub zero_point: tensor([64])
QuantStub dtype: torch.quint8


In [14]:
# Get the quantized weight of the convolution operation
print("Quantized weight (qint8): ", quantized_model.layer1[0].conv1.weight())
print("Quantized scale of weight: ", quantized_model.layer1[0].conv1.weight().q_per_channel_scales())
print("Quantized zero point of weight: ", quantized_model.layer1[0].conv1.weight().q_per_channel_zero_points())
print("Quantized scale of output: ", quantized_model.layer1[0].conv1.scale)
print("Quantized zero point of output: ", quantized_model.layer1[0].conv1.zero_point)

Quantized weight (qint8):  tensor([[[[-0.1448, -0.2414, -0.1448],
          [-0.3041, -0.5503, -0.4489],
          [-0.0483, -0.1883, -0.2124]],

         [[-0.0821, -0.2800, -0.2172],
          [-0.4151, -0.4586, -0.4972],
          [-0.4248, -0.4924, -0.1738]],

         [[-0.1014, -0.2269, -0.1496],
          [-0.4779, -0.4586, -0.6179],
          [-0.2414, -0.4055, -0.2896]],

         ...,

         [[-0.0145, -0.0338, -0.0917],
          [-0.0193,  0.0048,  0.0000],
          [ 0.0772,  0.0965,  0.0579]],

         [[ 0.0000,  0.0097, -0.0917],
          [ 0.0772,  0.1062,  0.0145],
          [ 0.0579,  0.0531, -0.0434]],

         [[ 0.0000, -0.0434, -0.0241],
          [ 0.0145,  0.0483, -0.0338],
          [-0.0241, -0.0290, -0.0338]]],


        [[[-0.1820,  0.0644,  0.0644],
          [-0.1596, -0.0896, -0.0364],
          [-0.0420, -0.1568, -0.1428]],

         [[-0.0588, -0.0084,  0.0112],
          [-0.0336,  0.0588, -0.0588],
          [-0.1008, -0.0504, -0.0700]],

    

In [15]:
# Get the parameters of batch normalization
print("BatchNorm running mean (float32): ", quantized_model.layer1[0].bn1.running_mean)
print("BatchNorm running var (float32): ", quantized_model.layer1[0].bn1.running_var)
print("BatchNorm scale (float32): ", quantized_model.layer1[0].bn1.weight)
print("BatchNorm bias (float32): ", quantized_model.layer1[0].bn1.bias)

BatchNorm running mean (float32):  tensor([-2.2942,  1.3833, -1.1609, -0.4893, -0.9163, -0.8867, -1.3708, -1.6399,
        -0.6377, -0.4159, -0.5340, -0.4263, -0.3651, -0.2259, -0.7256, -2.3012])
BatchNorm running var (float32):  tensor([4.8488, 1.3157, 3.2179, 2.2737, 1.1916, 1.6326, 4.0242, 1.1698, 2.4731,
        2.8714, 1.3212, 1.4982, 1.1879, 1.0504, 1.1262, 2.2886])
BatchNorm scale (float32):  Parameter containing:
tensor([1.5055, 1.0621, 1.0520, 1.0123, 0.7473, 1.0718, 1.0267, 0.7904, 1.0714,
        0.7943, 1.0628, 0.8785, 0.8526, 0.8969, 0.9422, 1.0002],
       requires_grad=True)
BatchNorm bias (float32):  Parameter containing:
tensor([-0.0278, -0.3303, -0.1472, -0.0698,  0.0354, -0.2506, -0.2725,  0.4111,
        -0.1301,  0.0935, -0.0696, -0.1104,  0.0997,  0.1137,  0.1260,  0.0824],
       requires_grad=True)


In [16]:
# Get the quantized weight of the feed forward operation
print("Quantized weight (qint8): ", quantized_model.fc.weight())
print("Quantized scale: ", quantized_model.fc.weight().q_per_channel_scales())
print("Quantized zero point: ", quantized_model.fc.weight().q_per_channel_zero_points())
print("Quantized bias (float32): ", quantized_model.fc.bias())
print("Quantized scale of output: ", quantized_model.fc.scale)
print("Quantized zero point of output: ", quantized_model.fc.zero_point)

Quantized weight (qint8):  tensor([[-0.2051,  0.1172, -0.3662,  0.1318,  0.5347, -0.1758,  0.2930, -0.4907,
          0.1611, -0.0806,  0.0586, -0.9375,  0.3516,  0.3955,  0.0513, -0.5493,
          0.3516, -0.6739,  0.2271, -0.5127,  0.1758,  0.3150, -0.4688, -0.0146,
         -0.4175, -0.3150,  0.3516,  0.0073, -0.3589,  0.5640, -0.1025, -0.5347],
        [ 0.7421, -0.5598,  0.7296, -0.7987, -0.3082, -0.5660, -0.7987, -0.5094,
         -0.7359,  0.5094, -0.3648, -0.5535, -0.1006,  0.4591, -0.3585,  0.3585,
          0.2893, -0.4025,  0.0126,  0.0314, -0.1447, -0.6164,  0.0692,  0.3711,
         -0.6226,  0.4591,  0.2075,  0.2767, -0.1950, -0.3774,  0.2956,  0.0629],
        [-0.3987, -0.1213, -0.1502,  0.5432,  0.0867, -0.7396,  0.1734,  0.4334,
         -0.3698, -0.0231,  0.3698,  0.1040, -0.2889,  0.1560,  0.4449, -0.4912,
         -0.4449,  0.3352, -0.3236, -0.6010,  0.1445,  0.3929, -0.4334,  0.2196,
          0.1849,  0.1213, -0.3178, -0.7050,  0.0058,  0.1734, -0.0693,  0.2138]